# Logistic Regression Example (Wine Quality Dataset)

Here it is demonstrated how to use the `LogisticRegression` module from the CMOR-438 library to classify wine quality.

**Goal: Predict whether a wine is High Quality (score ≥ 7) based on its physicochemical properties.**

- **Class 0:** Low or Mid quality (score 3–6)
- **Class 1:** High quality (score 7–8)

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# NOTEBOOK_DIR resolves to the folder this notebook lives in.
# Jupyter sets the working directory to wherever it was launched from,
# so we use __file__ would not work — os.path.abspath('') is the correct
# approach for Jupyter notebooks.
# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/supervised/ at the repo root,
# which is four levels up from examples/supervised/<algo>/
SRC_SUP  = os.path.join(REPO_ROOT, 'src', 'supervised')
sys.path.insert(0, SRC_SUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from logistic_regression import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")
print(f"Quality distribution:\n{wine['quality'].value_counts().sort_index().to_string()}")

## 2. Preprocessing

Create a binary target, standardise features, and split 80/20.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_bin = (wine['quality'].values >= 7).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

print(f"Training samples: {X_tr.shape[0]}  |  High quality in train: {y_tr.sum()}")
print(f"Test samples:     {X_te.shape[0]}  |  High quality in test:  {y_te.sum()}")

## 3. Train

In [ ]:
log = LogisticRegression(learning_rate=0.1, n_iterations=600, l2=0.01)
log.fit(X_tr, y_tr)
print(f'Accuracy: {log.accuracy(X_te, y_te):.4f}')

## 4. Results and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log.loss_history_, color='crimson', lw=1.5)
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].set_title('Logistic Regression Loss Curve', fontweight='bold')

probs = log.predict_proba(X_te)
for label, color, name in zip([0,1],['steelblue','darkorange'],['Low/Mid','High Quality']):
    axes[1].hist(probs[y_te==label], bins=25, alpha=0.6, color=color, label=name)
axes[1].axvline(0.5, color='red', linestyle='--', lw=1.5, label='Threshold')
axes[1].set_xlabel('P(High Quality)'); axes[1].set_ylabel('Count')
axes[1].set_title('Predicted Probability Distribution', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Analysis

**Result: 86.9% test accuracy** on the High Quality (score ≥ 7) vs rest binary classification task.

This is strong performance given the class imbalance — only 32 of the 229 test samples (14%) are High Quality wines. A model that always predicts "not high quality" would achieve 86% accuracy trivially. The probability distribution plot is the key diagnostic here: if the model is genuinely learning and not just exploiting the imbalance, the two class histograms (High Quality and Low/Mid) should be visually separated, with High Quality wines receiving higher probabilities on average.

**The loss curve** decreasing smoothly to a plateau confirms gradient descent converged without oscillation. 600 iterations with learning rate 0.1 is well-matched to this problem.

**L2 regularisation (l2=0.01)** has a mild effect here — the dataset is not high-dimensional enough for severe overfitting, but it stabilises training.

**Key takeaway:** Logistic Regression achieves competitive accuracy with very fast training. Its limitation is that it draws a single linear decision boundary in the 11-dimensional feature space — wines with complex, non-linear quality signatures will be misclassified. The MLP and ensemble methods address this limitation.